# RYZ2002 - Yapay Zeka Uygulamaları Dersi Dönem Sonu Projesi
## Akıllı Güvenlik, Nesne Takibi ve Yasak Bölge İhlal Tespit Sistemi

**Öğrenci Bilgileri:**
- **Adı Soyadı:** Şevval [Soyadınızı Yazın]
- **Öğrenci Numarası:** [Öğrenci Numarasını Yazın]
- **Ders Grubu:** RYZ2002 Yapay Zeka Uygulamaları
- **Teslim Tarihi:** 1 Haziran 2026

Bu notebook, ders kapsamında öğrendiğiniz **Görüntü İşleme** ve **Yapay Zeka / Nesne Takibi** konularını bir araya getiren interaktif **Akıllı Güvenlik ve Yasak Bölge İhlal Tespit** uygulamasını Google Colab üzerinde çalıştırmak için tasarlanmıştır.

### 🎬 Proje Özellikleri:
1. **Görüntü İşleme (OpenCV)**: Arayüzden anlık olarak seçilebilen Canny Kenar Algılama, Grayscale, Gaussian Blur ve Eşikleme filtreleri.
2. **Yapay Zeka (YOLOv8)**: Nesne tespiti, sınıflandırma ve yasaklı bölge ihlali analizi.
3. **Yedek Algılayıcı (OpenCV Haar Cascade)**: Eğer sistemde `ultralytics` kütüphanesi yüklenemezse veya GPU bulunamazsa, sistem çökmek yerine OpenCV Yüz Algılama moduna geçerek çalışmaya devam eder.
4. **İnteraktif Çokgen Yasak Bölge**: Canlı görüntü üzerine tıklayarak kendi çokgen ihlal alanınızı çizebilirsiniz.
5. **Canlı İstatistik & Grafik**: Chart.js ile çizilen anlık nesne grafiği ve ihlal günlükleri (logs).

--- 

### 🚀 Google Colab Çalıştırma Talimatları:
1. Yukarıdan aşağıya tüm hücreleri sırayla çalıştırın.
2. Son hücreyi çalıştırdığınızda belirecek olan **mavi tünel bağlantısına** (Google Colab Proxy Linki) tıklayın.
3. Açılan web sayfasında tarayıcınızın kamera iznini onaylayın. Ekrana tıklayarak yasak bölgenizi çizin ve sistemi test edin!

### 📦 1. Bağımlılıkların Kurulması ve Klasör Yapısının Hazırlanması

In [ ]:
# Gerekli kütüphaneleri kur
!pip install ultralytics flask flask-cors waitress mediapipe

In [ ]:
# Gerekli şablon ve statik klasörleri oluştur
!mkdir -p templates static/css static/js

### 🛠️ 2. Kaynak Kod Dosyalarının Oluşturulması
Aşağıdaki hücreler projenin backend ve frontend kodlarını otomatik olarak oluşturur.

#### 📝 Flask Backend Kodları (`app.py`)
Bu hücreyi çalıştırarak dosyayı Colab çalışma alanına yazdırın.

In [ ]:
%%writefile app.py
import os
import base64
import time
import json
from datetime import datetime
import numpy as np
import cv2
from flask import Flask, request, jsonify, render_template
from flask_cors import CORS

# Flask uygulamasını başlat
app = Flask(__name__, template_folder='templates', static_folder='static')
CORS(app)

# Global değişkenler ve model yükleme
HAS_YOLO = False
yolo_model = None

try:
    from ultralytics import YOLO
    # YOLOv8n (nano) modelini yükle, bulutta hızlı çalışır
    # Colab'de GPU varsa otomatik kullanır
    yolo_model = YOLO('yolov8n.pt')
    HAS_YOLO = True
    print("[INFO] YOLOv8 modeli başarıyla yüklendi.")
except Exception as e:
    print(f"[WARNING] YOLOv8 yüklenemedi (ultralytics bulunamadı veya hata oluştu): {e}")
    print("[INFO] Sistem OpenCV Haar Cascade (Yüz Algılama) yedek moduna geçiyor.")

# Haar Cascade Yedek Yüz Algılayıcı (YOLO yüklü değilse veya yedek olarak kullanılacak)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Mediapipe El Algılayıcı
HAS_MEDIAPIPE = False
try:
    import mediapipe as mp
    mp_hands = mp.solutions.hands
    hands_detector = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )
    HAS_MEDIAPIPE = True
    print("[INFO] Mediapipe Hands başarıyla yüklendi.")
except Exception as e:
    print(f"[WARNING] Mediapipe Hands yüklenemedi: {e}")

# İhlal logları ve istatistikleri (Bellekte tutulur)
event_logs = []
cumulative_counts = {}

def apply_opencv_filter(image, filter_type):
    """
    Kullanıcının seçtiği OpenCV görüntü işleme filtresini uygular.
    """
    if filter_type == 'grayscale':
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    elif filter_type == 'canny':
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        # Gürültü azaltma ve kenar algılama
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)
        canny = cv2.Canny(blurred, 50, 150)
        return cv2.cvtColor(canny, cv2.COLOR_GRAY2BGR)
    elif filter_type == 'blur':
        return cv2.GaussianBlur(image, (15, 15), 0)
    elif filter_type == 'threshold':
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        _, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
        return cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)
    return image

@app.route('/')
def index():
    """Ana sayfa yönlendirmesi"""
    return render_template('index.html', has_yolo=HAS_YOLO)

@app.route('/process_frame', methods=['POST'])
def process_frame():
    t_start = time.time()
    
    # JSON verisini al
    data = request.get_json(silent=True) or {}
    
    # Base64 görüntüyü al ve çöz
    img_data_b64 = data.get('image', '')
    if not img_data_b64:
        return jsonify({'error': 'Görüntü verisi bulunamadı'}), 400
        
    if ',' in img_data_b64:
        img_data_b64 = img_data_b64.split(',')[1]
        
    try:
        img_bytes = base64.b64decode(img_data_b64)
        np_arr = np.frombuffer(img_bytes, np.uint8)
        frame = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
    except Exception as e:
        return jsonify({'error': f'Görüntü çözülemedi: {str(e)}'}), 400
        
    if frame is None:
        return jsonify({'error': 'Görüntü çözülemedi (Boş kare)'}), 400

    # Ayna modu kontrolü (Görüntüyü Yatay Çevir)
    mirror = data.get('mirror', False)
    if mirror:
        frame = cv2.flip(frame, 1)

    # Parametreleri al
    polygon_ratios = data.get('polygon', [])
    filter_type = data.get('filter', 'none')
    conf_threshold = float(data.get('confidence', 0.25))
    target_classes = data.get('target_classes', ['person', 'car', 'dog', 'cat', 'bicycle', 'hand'])

    height, width = frame.shape[:2]
    
    # Çokgen (Yasaklı Bölge) piksel koordinatlarını hesapla
    polygon_pts = []
    if polygon_ratios:
        polygon_pts = np.array([[int(pt[0] * width), int(pt[1] * height)] for pt in polygon_ratios], dtype=np.int32)
        # OpenCV işlemleri için format
        if len(polygon_pts) > 0:
            polygon_pts = polygon_pts.reshape((-1, 1, 2))

    # Yasaklı bölge maskesi oluştur (En ufak temas/kesişimi tespit etmek için)
    polygon_mask = None
    if len(polygon_pts) >= 3:
        polygon_mask = np.zeros((height, width), dtype=np.uint8)
        cv2.fillPoly(polygon_mask, [polygon_pts], 255)

    detections = []
    intrusion_detected = False
    class_counts = {}

    # 1. YOLOv8 ile Tahmin ve Takip Aşaması
    if HAS_YOLO:
        results = yolo_model(frame, conf=conf_threshold, verbose=False)
        
        if results and len(results) > 0:
            result = results[0]
            boxes = result.boxes
            
            for box in boxes:
                cls_id = int(box.cls[0].item())
                conf = float(box.conf[0].item())
                
                # Sınıf adını al
                cls_name = yolo_model.names[cls_id]
                
                # Eğer sınıf hedef listemizde yoksa atla
                if cls_name not in target_classes:
                    continue
                
                # Bounding box koordinatları [x1, y1, x2, y2]
                xyxy = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = map(int, xyxy)
                
                # Kesişim maskesi ile hassas ihlal denetimi (Gereksiz boş alanların ihlal vermesini önlemek için daraltılmış kutu)
                is_inside = False
                if polygon_mask is not None:
                    w = x2 - x1
                    h = y2 - y1
                    cx = (x1 + x2) // 2
                    cy = (y1 + y2) // 2
                    
                    if cls_name == 'person':
                        # İnsan için genişliği %40'a daralt (kolları ve boşlukları ele, gövdeyi kontrol et)
                        sx1 = max(0, int(cx - 0.20 * w))
                        sx2 = min(width - 1, int(cx + 0.20 * w))
                        sy1 = y1
                        sy2 = y2
                    else:
                        # Diğer nesneler için genel kutuyu %70'e daralt
                        sx1 = max(0, int(cx - 0.35 * w))
                        sx2 = min(width - 1, int(cx + 0.35 * w))
                        sy1 = max(0, int(cy - 0.35 * h))
                        sy2 = min(height - 1, int(cy + 0.35 * h))
                    
                    box_mask = np.zeros((height, width), dtype=np.uint8)
                    cv2.rectangle(box_mask, (sx1, sy1), (sx2, sy2), 255, -1)
                    overlap = cv2.bitwise_and(polygon_mask, box_mask)
                    if np.any(overlap > 0):
                        is_inside = True
                        intrusion_detected = True
                
                px, py = (x1 + x2) // 2, (y1 + y2) // 2
                
                # Sınıf sayısını arttır
                class_counts[cls_name] = class_counts.get(cls_name, 0) + 1
                
                detections.append({
                    'class': 'person' if cls_name == 'person' else cls_name,
                    'confidence': round(conf, 2),
                    'box': [x1, y1, x2, y2],
                    'is_intruder': is_inside,
                    'center': [px, py]
                })

    # 2. Mediapipe ile El Algılama (YOLOv8'de el sınıfı olmadığı için paralel çalışır)
    if HAS_MEDIAPIPE and 'hand' in target_classes:
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results_hands = hands_detector.process(rgb_frame)
        if results_hands.multi_hand_landmarks:
            for hand_landmarks in results_hands.multi_hand_landmarks:
                # Elin sınır koordinatlarını (bounding box) hesapla
                x_max = 0.0
                y_max = 0.0
                x_min = 1.0
                y_min = 1.0
                for lm in hand_landmarks.landmark:
                    x_min = min(x_min, lm.x)
                    y_min = min(y_min, lm.y)
                    x_max = max(x_max, lm.x)
                    y_max = max(y_max, lm.y)
                
                # Normalize koordinatları piksel koordinatlarına dönüştür
                x1 = max(0, int(x_min * width))
                y1 = max(0, int(y_min * height))
                x2 = min(width - 1, int(x_max * width))
                y2 = min(height - 1, int(y_max * height))
                
                # El kutusu ile yasaklı bölge kesişim denetimi (Hafifçe daraltılmış %85 boyut)
                is_inside = False
                if polygon_mask is not None:
                    w = x2 - x1
                    h = y2 - y1
                    cx = (x1 + x2) // 2
                    cy = (y1 + y2) // 2
                    
                    sx1 = max(0, int(cx - 0.425 * w))
                    sx2 = min(width - 1, int(cx + 0.425 * w))
                    sy1 = max(0, int(cy - 0.425 * h))
                    sy2 = min(height - 1, int(cy + 0.425 * h))
                    
                    box_mask = np.zeros((height, width), dtype=np.uint8)
                    cv2.rectangle(box_mask, (sx1, sy1), (sx2, sy2), 255, -1)
                    overlap = cv2.bitwise_and(polygon_mask, box_mask)
                    if np.any(overlap > 0):
                        is_inside = True
                        intrusion_detected = True
                
                px, py = (x1 + x2) // 2, (y1 + y2) // 2
                cls_name = 'hand'
                class_counts[cls_name] = class_counts.get(cls_name, 0) + 1
                
                detections.append({
                    'class': 'El (Hand)',
                    'confidence': 0.90, # Mediapipe el için sabit yüksek güven
                    'box': [x1, y1, x2, y2],
                    'is_intruder': is_inside,
                    'center': [px, py]
                })

    # 3. Yüz Algılama (OpenCV Haar Cascade) - Yakın çekimde YOLOv8 insanı kaçırırsa yüz üzerinden yakalamak için paralel çalışır
    if 'person' in target_classes:
        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray_frame, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
        for (x, y, w, h) in faces:
            x1, y1, x2, y2 = x, y, x + w, y + h
            
            # Eğer YOLO zaten bu yüzü kapsayan bir insan algıladıysa mükerrer algılamayı önleyelim
            is_duplicate = False
            for det in detections:
                if det['class'] == 'person':
                    px1, py1, px2, py2 = det['box']
                    if px1 <= x + w//2 <= px2 and py1 <= y + h//2 <= py2:
                        is_duplicate = True
                        break
            
            if is_duplicate:
                continue
                
            # Yüz kutusu ile yasaklı bölge kesişim denetimi (Hafifçe daraltılmış %80 boyut)
            is_inside = False
            if polygon_mask is not None:
                w_face = x2 - x1
                h_face = y2 - y1
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2
                
                sx1 = max(0, int(cx - 0.40 * w_face))
                sx2 = min(width - 1, int(cx + 0.40 * w_face))
                sy1 = max(0, int(cy - 0.40 * h_face))
                sy2 = min(height - 1, int(cy + 0.40 * h_face))
                
                box_mask = np.zeros((height, width), dtype=np.uint8)
                cv2.rectangle(box_mask, (sx1, sy1), (sx2, sy2), 255, -1)
                overlap = cv2.bitwise_and(polygon_mask, box_mask)
                if np.any(overlap > 0):
                    is_inside = True
                    intrusion_detected = True
            
            px, py = (x1 + x2) // 2, (y1 + y2) // 2
            cls_name = 'person'
            class_counts[cls_name] = class_counts.get(cls_name, 0) + 1
            
            detections.append({
                'class': 'person', # Kutucuğun hemen üstünde person yazması için
                'confidence': 0.85,
                'box': [x1, y1, x2, y2],
                'is_intruder': is_inside,
                'center': [px, py]
            })

    # 4. Görüntü İşleme Filtresini Uygula (Görsel Katman)
    processed_frame = apply_opencv_filter(frame.copy(), filter_type)

    # 5. Görsel Overlay ve Çizimler
    zone_color = (0, 0, 255) if intrusion_detected else (0, 255, 0)
    zone_thickness = 3 if intrusion_detected else 2
    
    if len(polygon_pts) > 0:
        cv2.polylines(processed_frame, [polygon_pts], isClosed=True, color=zone_color, thickness=zone_thickness)
        overlay = processed_frame.copy()
        cv2.fillPoly(overlay, [polygon_pts], zone_color)
        alpha = 0.15 if not intrusion_detected else 0.3
        cv2.addWeighted(overlay, alpha, processed_frame, 1 - alpha, 0, processed_frame)

    # Algılanan nesneleri çiz
    for det in detections:
        x1, y1, x2, y2 = det['box']
        is_intruder = det['is_intruder']
        cls_name = det['class']
        conf = det['confidence']
        
        box_color = (0, 0, 255) if is_intruder else (255, 0, 128)
        cv2.rectangle(processed_frame, (x1, y1), (x2, y2), box_color, 2)
        
        label = f"{cls_name} %{int(conf*100)}"
        if is_intruder:
            label += " [IHLAL!]"
            
        (w_text, h_text), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(processed_frame, (x1, y1 - 20), (x1 + w_text, y1), box_color, -1)
        cv2.putText(processed_frame, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        cv2.circle(processed_frame, tuple(det['center']), 4, (0, 255, 255), -1)

    # İhlal Durumu Banner'ı
    if intrusion_detected:
        cv2.rectangle(processed_frame, (0, 0), (width, 40), (0, 0, 255), -1)
        cv2.putText(processed_frame, "!!! YASAKLI BOLGE IHLALI !!!", (width // 2 - 150, 26), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
        
        from datetime import timezone, timedelta
        tr_tz = timezone(timedelta(hours=3))
        now_str = datetime.now(tr_tz).strftime("%H:%M:%S")
        msg = f"Yasaklı Bölge İhlali Tespit Edildi!"
        
        if not event_logs or (time.time() - event_logs[-1]['time_raw'] > 1.5):
            event_logs.append({
                'timestamp': now_str,
                'message': msg,
                'type': 'danger',
                'time_raw': time.time()
            })
            cumulative_counts['violations'] = cumulative_counts.get('violations', 0) + 1

    # Bilgi Banner'ı
    status_text = "YOLOv8 Aktif" if HAS_YOLO else "OpenCV Yedek Mod (Yüz Algilama)"
    status_color = (0, 255, 0) if HAS_YOLO else (0, 165, 255)
    cv2.putText(processed_frame, status_text, (10, height - 15), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, status_color, 1, cv2.LINE_AA)

    # FPS hesaplama
    fps = round(1.0 / (time.time() - t_start), 1)

    # İşlenmiş görüntüyü Base64'e dönüştür
    _, buffer = cv2.imencode('.jpg', processed_frame)
    processed_b64 = base64.b64encode(buffer).decode('utf-8')
    processed_b64_src = f"data:image/jpeg;base64,{processed_b64}"

    if len(event_logs) > 15:
        event_logs.pop(0)

    return jsonify({
        'image': processed_b64_src,
        'intrusion_detected': intrusion_detected,
        'detections': [{'class': d['class'], 'confidence': d['confidence'], 'is_intruder': d['is_intruder']} for d in detections],
        'counts': class_counts,
        'cumulative_violations': cumulative_counts.get('violations', 0),
        'fps': fps,
        'logs': [{'timestamp': log['timestamp'], 'message': log['message'], 'type': log['type']} for log in reversed(event_logs)]
    })

@app.route('/reset_stats', methods=['POST'])
def reset_stats():
    global event_logs, cumulative_counts
    event_logs = []
    cumulative_counts = {'violations': 0}
    return jsonify({'status': 'success', 'message': 'İstatistikler sıfırlandı.'})

if __name__ == '__main__':
    try:
        from waitress import serve
        print("[INFO] Üretim WSGI Sunucusu (Waitress) başlatılıyor... Port: 5000")
        serve(app, host='0.0.0.0', port=5000)
    except ImportError:
        print("[WARNING] Waitress modülü yüklenemedi. Flask geliştirme sunucusu kullanılıyor...")
        app.run(host='0.0.0.0', port=5000, debug=False)


#### 📝 HTML Tasarımı (`templates/index.html`)
Bu hücreyi çalıştırarak dosyayı Colab çalışma alanına yazdırın.

In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="tr">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <meta name="description" content="RYZ2002 Yapay Zeka Uygulamaları Dersi Dönem Sonu Projesi - Akıllı Güvenlik, Nesne Takibi ve Yasak Bölge İhlal Tespit Sistemi.">
    <title>RYZ2002 - Akıllı Güvenlik ve İhlal Tespit Sistemi</title>
    <!-- Custom CSS Link -->
    <link rel="stylesheet" href="{{ url_for('static', filename='css/style.css') }}">
</head>
<body id="app-body">
    <!-- Top Navigation / Header -->
    <header>
        <div class="logo-container">
            <div class="logo-icon">
                <!-- SVG Shield Logo -->
                <svg viewBox="0 0 24 24" xmlns="http://www.w3.org/2000/svg">
                    <path d="M12 22s8-4 8-10V5l-8-3-8 3v7c0 6 8 10 8 10z"/>
                </svg>
            </div>
            <div>
                <h1>Akıllı Güvenlik & İhlal Tespit Sistemi</h1>
            </div>
        </div>
        <div class="header-badge">
            RYZ2002 Dönem Sonu Projesi
        </div>
    </header>

    <!-- Main Content Grid -->
    <main>
        <!-- Left Section: Live Feed and Settings Controls -->
        <div class="video-section">
            <!-- Video Display Panel -->
            <div class="video-container glass-panel" id="video-wrapper">
                <!-- Hidden HTML Video for capturing frames in browser -->
                <video id="webcam-video" autoplay playsinline muted style="display:none;"></video>
                
                <!-- Main canvas where user draws the zone and webcam is rendered initially -->
                <canvas id="canvas-overlay"></canvas>
                
                <!-- Displayed image from flask containing the processed frame -->
                <img id="video-stream-img" class="video-stream-img" alt="İşlenmiş Yapay Zeka Akışı">
                
                <!-- Webcam Offline Placeholder -->
                <div class="video-placeholder" id="video-placeholder">
                    <div class="placeholder-icon">
                        <svg width="36" height="36" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" class="lucide lucide-video-off">
                            <path d="M10.66 6H14a2 2 0 0 1 2 2v2.34"/>
                            <path d="m22 8-6 4 6 4V8z"/>
                            <path d="M2 2l20 20"/>
                            <path d="M21.5 21.5H3a2 2 0 0 1-2-2V8a2 2 0 0 1 2-2h3.34"/>
                        </svg>
                    </div>
                    <h2>Kamera Bağlantısı Yok</h2>
                    <p>Yapay zeka analizini başlatmak için aşağıdaki "Kamerayı Başlat" butonuna tıklayın.</p>
                </div>
            </div>

            <!-- Dashboard Interaction Buttons -->
            <div class="btn-group">
                <button class="btn btn-primary" id="btn-toggle-cam">
                    <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" class="lucide lucide-play"><polygon points="5 3 19 12 5 21 5 3"/></svg>
                    Kamerayı Başlat
                </button>
                <button class="btn btn-secondary" id="btn-clear-zone">
                    <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" class="lucide lucide-trash-2"><path d="M3 6h18"/><path d="M19 6v14c0 1-1 2-2 2H7c-1 0-2-1-2-2V6"/><path d="M8 6V4c0-1 1-2 2-2h4c1 0 2 1 2 2v2"/></svg>
                    Bölgeyi Temizle
                </button>
                <button class="btn btn-danger" id="btn-reset-stats">
                    <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" class="lucide lucide-rotate-ccw"><path d="M3 12a9 9 0 1 0 9-9 9.75 9.75 0 0 0-6.74 2.74L3 8"/><path d="M3 3v5h5"/></svg>
                    İstatistikleri Sıfırla
                </button>
            </div>

            <!-- Control Settings Panel -->
            <div class="controls-grid">
                <!-- Image Processing Settings -->
                <div class="control-card glass-panel">
                    <h3 class="control-title">
                        <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" class="lucide lucide-sliders-horizontal"><line x1="21" x2="14" y1="4" y2="4"/><line x1="10" x2="3" y1="4" y2="4"/><line x1="21" x2="12" y1="12" y2="12"/><line x1="8" x2="3" y1="12" y2="12"/><line x1="21" x2="16" y1="20" y2="20"/><line x1="12" x2="3" y1="20" y2="20"/><line x1="14" x2="14" y1="2" y2="6"/><line x1="8" x2="8" y1="10" y2="14"/><line x1="16" x2="16" y1="18" y2="22"/></svg>
                        Görüntü İşleme Filtreleri
                    </h3>
                    <div class="control-group">
                        <label for="filter-select">Görüntü İşleme Metodu</label>
                        <select id="filter-select">
                            <option value="none">Normal (Filtresiz)</option>
                            <option value="grayscale">Grayscale (Gri Tonlama)</option>
                            <option value="canny">Canny Edge (Kenar Algılama)</option>
                            <option value="blur">Gaussian Blur (Bulanıklaştırma)</option>
                            <option value="threshold">Binary Threshold (Eşikleme)</option>
                        </select>
                        <p style="font-size: 0.75rem; color: var(--text-muted); margin-top: 5px;">
                            Seçilen OpenCV filtresi, yapay zeka tespit kutularının altında eş zamanlı olarak uygulanır.
                        </p>
                    </div>
                </div>

                <!-- AI and Class Detection Settings -->
                <div class="control-card glass-panel">
                    <h3 class="control-title">
                        <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" class="lucide lucide-brain-circuit"><path d="M12 5a3 3 0 1 0-5.997.125 4 4 0 0 0-2.526 5.77 4 4 0 0 0 .556 6.588A4 4 0 1 0 12 18Z"/><path d="M9 13a3 3 0 0 1 3-3c1.657 0 3 1.343 3 3"/><path d="M12 10V3"/><path d="M12 18v3"/><path d="M16 13h5"/><path d="M5 13h2"/><path d="m19 9-2.5 2.5"/><path d="m19 17-2.5-2.5"/><path d="m5 9 2.5 2.5"/><path d="M17 13a3 3 0 0 0-3-3"/></svg>
                        Yapay Zeka Sınıf Ayarları
                    </h3>
                    <div class="control-group">
                        <label>YOLOv8 Güven Eşiği (Confidence)</label>
                        <div class="slider-container">
                            <input type="range" id="conf-slider" min="0.10" max="0.90" step="0.05" value="0.25">
                            <span class="slider-val" id="conf-val">0.25</span>
                        </div>
                        
                        <label style="margin-top: 8px;">Algılanacak Sınıflar</label>
                        <div class="checkbox-grid">
                            <label class="checkbox-container">
                                <input type="checkbox" class="class-checkbox" value="person" checked>
                                <span class="custom-checkmark"></span> İnsan
                            </label>
                            <label class="checkbox-container">
                                <input type="checkbox" class="class-checkbox" value="car" checked>
                                <span class="custom-checkmark"></span> Otomobil
                            </label>
                            <label class="checkbox-container">
                                <input type="checkbox" class="class-checkbox" value="dog" checked>
                                <span class="custom-checkmark"></span> Köpek
                            </label>
                            <label class="checkbox-container">
                                <input type="checkbox" class="class-checkbox" value="cat" checked>
                                <span class="custom-checkmark"></span> Kedi
                            </label>
                            <label class="checkbox-container">
                                <input type="checkbox" class="class-checkbox" value="bicycle">
                                <span class="custom-checkmark"></span> Bisiklet
                            </label>
                            <label class="checkbox-container">
                                <input type="checkbox" class="class-checkbox" value="hand" checked>
                                <span class="custom-checkmark"></span> El (Hand)
                            </label>
                            <label class="checkbox-container">
                                <input type="checkbox" id="toggle-audio" checked>
                                <span class="custom-checkmark"></span> Sesli Alarm
                            </label>
                            <label class="checkbox-container" style="grid-column: span 2;">
                                <input type="checkbox" id="toggle-mirror">
                                <span class="custom-checkmark"></span> Ayna Modu (Görüntüyü Çevir)
                            </label>
                        </div>
                    </div>
                </div>
            </div>
        </div>

        <!-- Right Section: Statistics, Live Charts, and Log History -->
        <div class="stats-section">
            <!-- System Security Status Banner -->
            <div class="status-panel glass-panel">
                <span class="status-label">Sistem Durumu:</span>
                <span class="status-badge status-secure" id="status-badge">
                    <span class="status-dot"></span>
                    <span id="status-text">Güvenli</span>
                </span>
            </div>

            <!-- Numerical Statistics Cards -->
            <div class="stats-grid">
                <div class="stat-card glass-panel" id="card-active-detect">
                    <span class="stat-value" id="val-active-detect">0</span>
                    <span class="stat-name">Aktif Tespit</span>
                </div>
                <div class="stat-card glass-panel" id="card-total-alert">
                    <span class="stat-value" id="val-total-alert">0</span>
                    <span class="stat-name">Toplam İhlal</span>
                </div>
                <div class="stat-card glass-panel">
                    <span class="stat-value" id="val-fps">0.0</span>
                    <span class="stat-name">İşlem Hızı (FPS)</span>
                </div>
            </div>

            <!-- Live Chart Panel -->
            <div class="chart-panel glass-panel">
                <h3 class="control-title" style="margin-bottom: 5px; border-bottom: none;">
                    <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" class="lucide lucide-trending-up"><polyline points="22 7 13.5 15.5 8.5 10.5 2 17"/><polyline points="16 7 22 7 22 13"/></svg>
                    Canlı Nesne Grafiği
                </h3>
                <div class="chart-container">
                    <canvas id="live-chart"></canvas>
                </div>
            </div>

            <!-- Alert Activity Log Panel -->
            <div class="logs-panel glass-panel">
                <h3 class="control-title" style="margin-bottom: 12px;">
                    <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" class="lucide lucide-bell-ring"><path d="M6 8a6 6 0 0 1 12 0c0 7 3 9 3 9H3s3-2 3-9"/><path d="M10.3 21a1.94 1.94 0 0 0 3.4 0"/><path d="M4 2C2.8 3.7 2 5.7 2 8"/><path d="M20 2c1.2 1.7 2 3.7 2 6"/></svg>
                    Güvenlik İhlal Günlüğü
                </h3>
                <ul class="logs-list" id="logs-list">
                    <li class="log-item" style="border-left-color: var(--color-secondary);">
                        <span class="log-time">--:--:--</span>
                        <span class="log-msg">Sistem başlatıldı. İzleme bekleniyor...</span>
                    </li>
                </ul>
            </div>
        </div>
    </main>

    <!-- Footer -->
    <footer>
        <p>RYZ2002 Yapay Zeka Uygulamaları Dersi Dönem Sonu Projesi &copy; 2026. <br>
           Google Colab üzerinde GPU hızlandırıcı ile çalışacak şekilde optimize edilmiştir.</p>
    </footer>

    <!-- CDN Libraries: Chart.js & Lucide Icons -->
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <script src="https://unpkg.com/lucide@latest"></script>
    
    <!-- Custom Frontend Javascript -->
    <script src="{{ url_for('static', filename='js/main.js') }}"></script>
</body>
</html>


#### 📝 CSS Arayüz Tasarımı (`static/css/style.css`)
Bu hücreyi çalıştırarak dosyayı Colab çalışma alanına yazdırın.

In [ ]:
%%writefile static/css/style.css
/* Import Google Fonts (Outfit and Inter) */
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=Outfit:wght@300;400;500;600;700;800&display=swap');

/* CSS Variables for design system */
:root {
    --bg-main: #0a0a14;
    --bg-card: rgba(20, 20, 35, 0.65);
    --border-color: rgba(120, 100, 255, 0.15);
    --border-glow: rgba(120, 100, 255, 0.3);
    
    --color-primary: #7d5fff;
    --color-primary-glow: rgba(125, 95, 255, 0.4);
    --color-secondary: #00ebc7;
    --color-secondary-glow: rgba(0, 235, 199, 0.4);
    
    --color-success: #05c46b;
    --color-success-glow: rgba(5, 196, 107, 0.3);
    --color-danger: #ff3f34;
    --color-danger-glow: rgba(255, 63, 52, 0.45);
    --color-warning: #ffc048;
    
    --text-main: #f3f3f7;
    --text-muted: #8e8ea8;
    --font-header: 'Outfit', sans-serif;
    --font-body: 'Inter', sans-serif;
    
    --transition-speed: 0.25s;
}

/* Screen Flash Alert Overlay */
.screen-flash-active {
    animation: edge-flash 1s infinite alternate;
}

@keyframes edge-flash {
    0% {
        box-shadow: inset 0 0 20px rgba(255, 63, 52, 0.2);
    }
    100% {
        box-shadow: inset 0 0 60px rgba(255, 63, 52, 0.7);
    }
}

/* Reset and Global Styles */
* {
    margin: 0;
    padding: 0;
    box-sizing: border-box;
}

body {
    background-color: var(--bg-main);
    color: var(--text-main);
    font-family: var(--font-body);
    min-height: 100vh;
    overflow-x: hidden;
    position: relative;
    background-image: 
        radial-gradient(circle at 10% 20%, rgba(125, 95, 255, 0.08) 0%, transparent 40%),
        radial-gradient(circle at 90% 80%, rgba(0, 235, 199, 0.08) 0%, transparent 40%);
}

/* Glassmorphism card container utility */
.glass-panel {
    background: var(--bg-card);
    backdrop-filter: blur(12px);
    -webkit-backdrop-filter: blur(12px);
    border: 1px solid var(--border-color);
    border-radius: 16px;
    box-shadow: 0 8px 32px 0 rgba(0, 0, 0, 0.4);
    transition: all var(--transition-speed) ease;
}

.glass-panel:hover {
    border-color: var(--border-glow);
    box-shadow: 0 8px 32px 0 rgba(125, 95, 255, 0.1);
}

/* App Header styling */
header {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 20px 40px;
    border-bottom: 1px solid rgba(255, 255, 255, 0.05);
    background: rgba(10, 10, 20, 0.8);
    backdrop-filter: blur(8px);
    position: sticky;
    top: 0;
    z-index: 100;
}

.logo-container {
    display: flex;
    align-items: center;
    gap: 12px;
}

.logo-icon {
    width: 38px;
    height: 38px;
    background: linear-gradient(135deg, var(--color-primary), var(--color-secondary));
    border-radius: 10px;
    display: flex;
    align-items: center;
    justify-content: center;
    box-shadow: 0 0 15px var(--color-primary-glow);
}

.logo-icon svg {
    width: 22px;
    height: 22px;
    fill: white;
}

h1 {
    font-family: var(--font-header);
    font-size: 1.5rem;
    font-weight: 700;
    letter-spacing: 0.5px;
    background: linear-gradient(to right, #ffffff, #a5b4fc);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

.header-badge {
    background: rgba(125, 95, 255, 0.15);
    border: 1px solid var(--color-primary);
    color: #c7d2fe;
    font-size: 0.75rem;
    padding: 6px 12px;
    border-radius: 20px;
    font-weight: 600;
    letter-spacing: 0.5px;
    box-shadow: 0 0 10px rgba(125, 95, 255, 0.2);
}

/* Layout Main Container */
main {
    max-width: 1600px;
    margin: 0 auto;
    padding: 30px;
    display: grid;
    grid-template-columns: 1.1fr 0.9fr;
    gap: 30px;
}

@media (max-width: 1024px) {
    main {
        grid-template-columns: 1fr;
    }
}

/* Left Column: Video & Controls */
.video-section {
    display: flex;
    flex-direction: column;
    gap: 20px;
}

.video-container {
    position: relative;
    width: 100%;
    aspect-ratio: 4/3;
    background-color: #050508;
    border-radius: 16px;
    overflow: hidden;
    display: flex;
    justify-content: center;
    align-items: center;
}

/* Canvas handles drawing zones */
#webcam-video, #canvas-overlay {
    position: absolute;
    top: 0;
    left: 0;
    width: 100%;
    height: 100%;
    object-fit: contain;
}

#canvas-overlay {
    z-index: 5;
    cursor: crosshair;
}

/* Placeholder when webcam is off */
.video-placeholder {
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    gap: 15px;
    text-align: center;
    color: var(--text-muted);
    z-index: 2;
    padding: 20px;
}

.placeholder-icon {
    font-size: 3rem;
    background: linear-gradient(135deg, var(--color-primary-glow), var(--color-secondary-glow));
    width: 80px;
    height: 80px;
    border-radius: 50%;
    display: flex;
    align-items: center;
    justify-content: center;
    animation: pulse 2s infinite ease-in-out;
}

@keyframes pulse {
    0%, 100% { transform: scale(1); opacity: 0.8; }
    50% { transform: scale(1.05); opacity: 1; }
}

.video-stream-img {
    width: 100%;
    height: 100%;
    object-fit: contain;
    z-index: 1;
    display: none;
}

/* Control Panel Grid */
.controls-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 20px;
}

@media (max-width: 600px) {
    .controls-grid {
        grid-template-columns: 1fr;
    }
}

.control-card {
    padding: 20px;
}

.control-title {
    font-family: var(--font-header);
    font-size: 1.1rem;
    font-weight: 600;
    margin-bottom: 15px;
    display: flex;
    align-items: center;
    gap: 8px;
    border-bottom: 1px solid rgba(255, 255, 255, 0.05);
    padding-bottom: 8px;
}

.control-group {
    display: flex;
    flex-direction: column;
    gap: 12px;
}

label {
    font-size: 0.85rem;
    font-weight: 500;
    color: var(--text-muted);
}

/* Select, range slider and inputs custom styling */
select {
    width: 100%;
    padding: 10px;
    background: rgba(20, 20, 30, 0.8);
    border: 1px solid rgba(255, 255, 255, 0.1);
    border-radius: 8px;
    color: var(--text-main);
    font-family: var(--font-body);
    font-size: 0.9rem;
    cursor: pointer;
    outline: none;
    transition: all var(--transition-speed);
}

select:focus {
    border-color: var(--color-primary);
    box-shadow: 0 0 8px rgba(125, 95, 255, 0.3);
}

/* Checkbox classes container */
.checkbox-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 8px;
}

.checkbox-container {
    display: flex;
    align-items: center;
    gap: 8px;
    cursor: pointer;
    font-size: 0.85rem;
    color: var(--text-main);
    user-select: none;
}

.checkbox-container input {
    display: none;
}

.custom-checkmark {
    width: 18px;
    height: 18px;
    border: 1px solid rgba(255, 255, 255, 0.2);
    border-radius: 4px;
    display: inline-block;
    position: relative;
    transition: all var(--transition-speed);
}

.checkbox-container input:checked + .custom-checkmark {
    background: var(--color-primary);
    border-color: var(--color-primary);
}

.checkbox-container input:checked + .custom-checkmark::after {
    content: "";
    position: absolute;
    left: 6px;
    top: 2px;
    width: 4px;
    height: 9px;
    border: solid white;
    border-width: 0 2px 2px 0;
    transform: rotate(45deg);
}

/* Range Slider styling */
.slider-container {
    display: flex;
    align-items: center;
    gap: 10px;
}

input[type="range"] {
    flex-grow: 1;
    -webkit-appearance: none;
    appearance: none;
    height: 6px;
    border-radius: 3px;
    background: rgba(255, 255, 255, 0.1);
    outline: none;
}

input[type="range"]::-webkit-slider-thumb {
    -webkit-appearance: none;
    appearance: none;
    width: 16px;
    height: 16px;
    border-radius: 50%;
    background: var(--color-primary);
    cursor: pointer;
    box-shadow: 0 0 8px var(--color-primary-glow);
    transition: transform 0.1s;
}

input[type="range"]::-webkit-slider-thumb:hover {
    transform: scale(1.2);
}

.slider-val {
    font-family: var(--font-header);
    font-size: 0.9rem;
    font-weight: 600;
    color: var(--color-secondary);
    min-width: 32px;
    text-align: right;
}

/* Button UI styling */
.btn-group {
    display: flex;
    gap: 10px;
    margin-top: 10px;
}

.btn {
    flex-grow: 1;
    padding: 12px 16px;
    border: none;
    border-radius: 8px;
    font-family: var(--font-header);
    font-weight: 600;
    font-size: 0.9rem;
    cursor: pointer;
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 8px;
    transition: all var(--transition-speed) ease;
}

.btn-primary {
    background: linear-gradient(135deg, var(--color-primary), #6344e3);
    color: white;
    box-shadow: 0 4px 15px rgba(125, 95, 255, 0.3);
}

.btn-primary:hover {
    transform: translateY(-2px);
    box-shadow: 0 6px 20px rgba(125, 95, 255, 0.5);
}

.btn-primary:active {
    transform: translateY(0);
}

.btn-secondary {
    background: rgba(255, 255, 255, 0.08);
    border: 1px solid rgba(255, 255, 255, 0.1);
    color: var(--text-main);
}

.btn-secondary:hover {
    background: rgba(255, 255, 255, 0.15);
    border-color: rgba(255, 255, 255, 0.2);
}

.btn-danger {
    background: rgba(255, 63, 52, 0.15);
    border: 1px solid var(--color-danger);
    color: #ff8a84;
}

.btn-danger:hover {
    background: var(--color-danger);
    color: white;
    box-shadow: 0 4px 15px var(--color-danger-glow);
}

/* Right Column: Statistics and Charts */
.stats-section {
    display: flex;
    flex-direction: column;
    gap: 20px;
}

/* System Status Ribbon and Indicator */
.status-panel {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 15px 25px;
}

.status-label {
    font-size: 0.9rem;
    font-weight: 500;
    color: var(--text-muted);
}

.status-badge {
    display: flex;
    align-items: center;
    gap: 10px;
    font-family: var(--font-header);
    font-weight: 700;
    font-size: 1.1rem;
    text-transform: uppercase;
}

.status-dot {
    width: 12px;
    height: 12px;
    border-radius: 50%;
}

.status-secure {
    color: var(--color-success);
}
.status-secure .status-dot {
    background-color: var(--color-success);
    box-shadow: 0 0 10px var(--color-success);
    animation: flash-green 1.5s infinite alternate;
}

.status-alarm {
    color: var(--color-danger);
}
.status-alarm .status-dot {
    background-color: var(--color-danger);
    box-shadow: 0 0 15px var(--color-danger);
    animation: flash-red 0.6s infinite alternate;
}

@keyframes flash-green {
    0% { opacity: 0.4; }
    100% { opacity: 1; }
}

@keyframes flash-red {
    0% { opacity: 0.3; transform: scale(0.9); }
    100% { opacity: 1; transform: scale(1.15); }
}

/* Stats Cards grid layout */
.stats-grid {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 15px;
}

.stat-card {
    padding: 15px;
    text-align: center;
    display: flex;
    flex-direction: column;
    gap: 8px;
}

.stat-value {
    font-family: var(--font-header);
    font-size: 1.8rem;
    font-weight: 800;
    background: linear-gradient(135deg, #ffffff, var(--text-muted));
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

.stat-card.alert-active .stat-value {
    background: linear-gradient(135deg, #ff8a84, var(--color-danger));
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    text-shadow: 0 0 15px rgba(255, 63, 52, 0.2);
}

.stat-name {
    font-size: 0.75rem;
    font-weight: 500;
    color: var(--text-muted);
    text-transform: uppercase;
    letter-spacing: 0.5px;
}

/* Chart Container panel */
.chart-panel {
    padding: 20px;
    display: flex;
    flex-direction: column;
    gap: 15px;
}

.chart-container {
    position: relative;
    width: 100%;
    height: 200px;
}

/* Event Logs list panel */
.logs-panel {
    padding: 20px;
    flex-grow: 1;
    display: flex;
    flex-direction: column;
    min-height: 250px;
}

.logs-list {
    list-style: none;
    overflow-y: auto;
    max-height: 200px;
    display: flex;
    flex-direction: column;
    gap: 8px;
    padding-right: 5px;
}

/* Custom Scrollbar for logs list */
.logs-list::-webkit-scrollbar {
    width: 6px;
}

.logs-list::-webkit-scrollbar-track {
    background: rgba(255, 255, 255, 0.02);
    border-radius: 3px;
}

.logs-list::-webkit-scrollbar-thumb {
    background: rgba(125, 95, 255, 0.2);
    border-radius: 3px;
}

.logs-list::-webkit-scrollbar-thumb:hover {
    background: rgba(125, 95, 255, 0.4);
}

.log-item {
    padding: 10px 14px;
    border-radius: 8px;
    font-size: 0.85rem;
    display: flex;
    justify-content: space-between;
    align-items: center;
    background: rgba(255, 255, 255, 0.03);
    border-left: 3px solid var(--text-muted);
    animation: slide-in 0.2s ease-out;
}

@keyframes slide-in {
    from { transform: translateY(-5px); opacity: 0; }
    to { transform: translateY(0); opacity: 1; }
}

.log-item.danger {
    background: rgba(255, 63, 52, 0.08);
    border-left-color: var(--color-danger);
    color: #ff8a84;
}

.log-time {
    font-family: var(--font-header);
    font-weight: 600;
    color: var(--text-muted);
    font-size: 0.8rem;
}

.log-msg {
    flex-grow: 1;
    margin-left: 10px;
}

/* Footer elements */
footer {
    max-width: 1600px;
    margin: 0 auto;
    padding: 30px;
    text-align: center;
    color: var(--text-muted);
    font-size: 0.8rem;
    border-top: 1px solid rgba(255, 255, 255, 0.03);
}

footer a {
    color: var(--color-primary);
    text-decoration: none;
    transition: color 0.2s;
}

footer a:hover {
    color: var(--color-secondary);
}


#### 📝 JavaScript Arayüz Kontrolörü (`static/js/main.js`)
Bu hücreyi çalıştırarak dosyayı Colab çalışma alanına yazdırın.

In [ ]:
%%writefile static/js/main.js
// DOM Elements
const video = document.getElementById('webcam-video');
const canvas = document.getElementById('canvas-overlay');
const ctx = canvas.getContext('2d');
const streamImg = document.getElementById('video-stream-img');
const placeholder = document.getElementById('video-placeholder');
const btnToggleCam = document.getElementById('btn-toggle-cam');
const btnClearZone = document.getElementById('btn-clear-zone');
const btnResetStats = document.getElementById('btn-reset-stats');

// Form & Control Inputs
const filterSelect = document.getElementById('filter-select');
const confSlider = document.getElementById('conf-slider');
const confVal = document.getElementById('conf-val');
const toggleAudio = document.getElementById('toggle-audio');
const toggleMirror = document.getElementById('toggle-mirror');
const classCheckboxes = document.querySelectorAll('.class-checkbox');

// Stats Elements
const statusBadge = document.getElementById('status-badge');
const statusText = document.getElementById('status-text');
const valActiveDetect = document.getElementById('val-active-detect');
const valTotalAlert = document.getElementById('val-total-alert');
const valFps = document.getElementById('val-fps');
const logsList = document.getElementById('logs-list');
const appBody = document.getElementById('app-body');

// State Variables
let isStreaming = false;
let stream = null;
let polygonPoints = []; // Stores normalized coordinates: [[x_ratio, y_ratio], ...]
let chart = null;
let animationFrameId = null;
let lastProcessedTime = 0;
let audioCtx = null;
let alarmInterval = null;
let lastAlarmTime = 0;

// Initialize Chart.js
function initChart() {
    const chartCtx = document.getElementById('live-chart').getContext('2d');
    chart = new Chart(chartCtx, {
        type: 'line',
        data: {
            labels: [], // Timestamps
            datasets: [{
                label: 'Aktif Nesne Sayısı',
                data: [],
                borderColor: '#7d5fff',
                backgroundColor: 'rgba(125, 95, 255, 0.15)',
                borderWidth: 2,
                fill: true,
                tension: 0.4,
                pointRadius: 2,
                pointHoverRadius: 5
            }]
        },
        options: {
            responsive: true,
            maintainAspectRatio: false,
            plugins: {
                legend: {
                    display: false
                }
            },
            scales: {
                x: {
                    grid: { display: false },
                    ticks: { color: '#8e8ea8', font: { size: 9 } }
                },
                y: {
                    min: 0,
                    suggestedMax: 5,
                    ticks: { stepSize: 1, color: '#8e8ea8', font: { size: 9 } },
                    grid: { color: 'rgba(255, 255, 255, 0.05)' }
                }
            }
        }
    });
}

// Adjust Canvas sizing to match visual container
function resizeCanvas() {
    canvas.width = canvas.clientWidth;
    canvas.height = canvas.clientHeight;
    drawPolygon();
}

// Draw the current polygon zone on the canvas overlay
function drawPolygon() {
    ctx.clearRect(0, 0, canvas.width, canvas.height);
    if (polygonPoints.length === 0) return;

    ctx.lineWidth = 2.5;
    ctx.strokeStyle = '#00ebc7'; // Neon cyan border
    ctx.fillStyle = 'rgba(0, 235, 199, 0.12)'; // Cyan semi-transparent fill

    ctx.beginPath();
    // Move to first point
    ctx.moveTo(polygonPoints[0][0] * canvas.width, polygonPoints[0][1] * canvas.height);
    
    // Draw dot for start point
    ctx.arc(polygonPoints[0][0] * canvas.width, polygonPoints[0][1] * canvas.height, 5, 0, 2 * Math.PI);
    ctx.fillStyle = '#7d5fff'; // Neon violet for starting point
    ctx.fill();

    ctx.beginPath();
    ctx.moveTo(polygonPoints[0][0] * canvas.width, polygonPoints[0][1] * canvas.height);
    for (let i = 1; i < polygonPoints.length; i++) {
        ctx.lineTo(polygonPoints[i][0] * canvas.width, polygonPoints[i][1] * canvas.height);
    }

    if (polygonPoints.length >= 3) {
        ctx.closePath();
        ctx.fillStyle = 'rgba(0, 235, 199, 0.12)';
        ctx.fill();
    }
    ctx.strokeStyle = '#00ebc7';
    ctx.stroke();

    // Draw other points
    for (let i = 1; i < polygonPoints.length; i++) {
        ctx.beginPath();
        ctx.arc(polygonPoints[i][0] * canvas.width, polygonPoints[i][1] * canvas.height, 4, 0, 2 * Math.PI);
        ctx.fillStyle = '#00ebc7';
        ctx.fill();
    }
}

// Handle canvas mouse click to define zone points
canvas.addEventListener('click', (e) => {
    if (!isStreaming) return;
    
    const rect = canvas.getBoundingClientRect();
    const x = e.clientX - rect.left;
    const y = e.clientY - rect.top;
    
    // Normalize coordinates (0.0 to 1.0) relative to canvas size
    const xRatio = x / canvas.width;
    const yRatio = y / canvas.height;
    
    polygonPoints.push([xRatio, yRatio]);
    drawPolygon();
});

// Clear Zone button listener
btnClearZone.addEventListener('click', () => {
    polygonPoints = [];
    drawPolygon();
    
    // Clear on canvas overlay
    ctx.clearRect(0, 0, canvas.width, canvas.height);
});

// Update confidence label on slider input
confSlider.addEventListener('input', (e) => {
    confVal.textContent = e.target.value;
});

// Play a high pitched alarm sound using the Web Audio API (no assets needed)
function playAlarmSound() {
    if (!toggleAudio.checked) return;
    
    // Throttle sound to prevent audio system crash (min 400ms between beeps)
    const now = Date.now();
    if (now - lastAlarmTime < 400) return;
    lastAlarmTime = now;

    if (!audioCtx) {
        audioCtx = new (window.AudioContext || window.webkitAudioContext)();
    }
    if (audioCtx.state === 'suspended') {
        audioCtx.resume();
    }

    const osc = audioCtx.createOscillator();
    const gain = audioCtx.createGain();
    
    osc.type = 'sawtooth';
    osc.frequency.setValueAtTime(950, audioCtx.currentTime); // Beep frequency
    
    gain.gain.setValueAtTime(0.12, audioCtx.currentTime);
    gain.gain.exponentialRampToValueAtTime(0.01, audioCtx.currentTime + 0.25);
    
    osc.connect(gain);
    gain.connect(audioCtx.destination);
    
    osc.start();
    osc.stop(audioCtx.currentTime + 0.25);
}

// Start/Stop Webcam
btnToggleCam.addEventListener('click', async () => {
    if (isStreaming) {
        stopWebcam();
    } else {
        await startWebcam();
    }
});

async function startWebcam() {
    try {
        // Try accessing user camera
        stream = await navigator.mediaDevices.getUserMedia({ 
            video: { 
                width: { ideal: 640 }, 
                height: { ideal: 480 },
                facingMode: 'user'
            } 
        });
        video.srcObject = stream;
        video.style.display = 'block';
        streamImg.style.display = 'block';
        placeholder.style.display = 'none';
        
        isStreaming = true;
        btnToggleCam.innerHTML = `
            <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" class="lucide lucide-square"><rect x="3" y="3" width="18" height="18" rx="2" ry="2"/></svg>
            İzlemeyi Durdur
        `;
        btnToggleCam.classList.remove('btn-primary');
        btnToggleCam.classList.add('btn-secondary');
        
        // Resize canvas to visual dimensions
        setTimeout(resizeCanvas, 500);
        
        // Start processing loop
        processFrameLoop();
    } catch (err) {
        console.error("Kamera başlatılamadı:", err);
        alert("Kamera erişimi sağlanamadı! Lütfen kamera izinlerini kontrol edin.");
    }
}

function stopWebcam() {
    isStreaming = false;
    if (stream) {
        stream.getTracks().forEach(track => track.stop());
    }
    video.srcObject = null;
    video.style.display = 'none';
    streamImg.style.display = 'none';
    placeholder.style.display = 'flex';
    
    btnToggleCam.innerHTML = `
        <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" class="lucide lucide-play"><polygon points="5 3 19 12 5 21 5 3"/></svg>
        Kamerayı Başlat
    `;
    btnToggleCam.classList.remove('btn-secondary');
    btnToggleCam.classList.add('btn-primary');
    
    // Clear canvas
    ctx.clearRect(0, 0, canvas.width, canvas.height);
    
    // Reset Status and Flash
    appBody.classList.remove('screen-flash-active');
    statusBadge.className = 'status-badge status-secure';
    statusText.textContent = 'Güvenli';
    valActiveDetect.textContent = '0';
    valFps.textContent = '0.0';
}

// Grab current video frame, convert to Base64, post to Flask API
async function processFrameLoop() {
    if (!isStreaming) return;

    const canvasTemp = document.createElement('canvas');
    canvasTemp.width = 480; // Gönderilen çözünürlük 480x360 yapıldı (transferi %44 hafifletir)
    canvasTemp.height = 360;
    const ctxTemp = canvasTemp.getContext('2d');
    
    // Draw current video frame to temp canvas
    ctxTemp.drawImage(video, 0, 0, canvasTemp.width, canvasTemp.height);
    
    // Convert to Base64 JPEG
    const base64Img = canvasTemp.toDataURL('image/jpeg', 0.55); // Kalite %55 yapıldı (veri boyutunu yarı yarıya düşürür)

    // Compile active classes
    const targetClasses = [];
    classCheckboxes.forEach(cb => {
        if (cb.checked) targetClasses.push(cb.value);
    });

    const payload = {
        image: base64Img,
        polygon: polygonPoints,
        filter: filterSelect.value,
        confidence: confSlider.value,
        target_classes: targetClasses,
        mirror: toggleMirror.checked
    };

    try {
        const response = await fetch('/process_frame', {
            method: 'POST',
            headers: {
                'Content-Type': 'application/json'
            },
            body: JSON.stringify(payload)
        });

        if (!response.ok) {
            throw new Error(`Server returned HTTP ${response.status}`);
        }

        const data = await response.json();
        
        // 1. Update Video Stream Image source with processed image
        streamImg.src = data.image;
        
        // 2. Update Status Indicators (Secure vs Alert)
        if (data.intrusion_detected) {
            statusBadge.className = 'status-badge status-alarm';
            statusText.textContent = 'İHLAL TESPİTİ!';
            appBody.classList.add('screen-flash-active');
            playAlarmSound();
        } else {
            statusBadge.className = 'status-badge status-secure';
            statusText.textContent = 'Güvenli';
            appBody.classList.remove('screen-flash-active');
        }

        // 3. Update Statistics
        valActiveDetect.textContent = data.detections.length;
        valTotalAlert.textContent = data.cumulative_violations;
        valFps.textContent = data.fps;

        // 4. Update Chart Data
        const timeNow = new Date().toLocaleTimeString([], { hour: '2-digit', minute: '2-digit', second: '2-digit' });
        updateChart(timeNow, data.detections.length);

        // 5. Update Log Activity List
        updateLogs(data.logs);

    } catch (error) {
        console.error("Frame işleme hatası:", error);
    }

    // Dynamic self-scheduling based on response loop (approx 10 FPS targets)
    if (isStreaming) {
        setTimeout(processFrameLoop, 30);
    }
}

// Update Chart.js data smoothly
function updateChart(label, value) {
    if (!chart) return;
    
    chart.data.labels.push(label);
    chart.data.datasets[0].data.push(value);
    
    // Limit data points to show a sliding window of the last 12 entries
    if (chart.data.labels.length > 12) {
        chart.data.labels.shift();
        chart.data.datasets[0].data.shift();
    }
    
    chart.update();
}

// Render Event Logs
function updateLogs(logs) {
    if (!logs) return;
    
    if (logs.length === 0) {
        logsList.innerHTML = `
            <li class="log-item" style="border-left-color: var(--color-secondary);">
                <span class="log-time">--:--:--</span>
                <span class="log-msg">Aktif alarm veya olay bulunmuyor.</span>
            </li>
        `;
        return;
    }

    let logsHTML = '';
    logs.forEach(log => {
        const itemClass = log.type === 'danger' ? 'danger' : '';
        logsHTML += `
            <li class="log-item ${itemClass}">
                <span class="log-time">${log.timestamp}</span>
                <span class="log-msg">${log.message}</span>
            </li>
        `;
    });
    logsList.innerHTML = logsHTML;
}

// Reset stats button listener
btnResetStats.addEventListener('click', async () => {
    try {
        const res = await fetch('/reset_stats', { method: 'POST' });
        const data = await res.json();
        if (data.status === 'success') {
            valTotalAlert.textContent = '0';
            updateLogs([]);
            // Clear chart
            if (chart) {
                chart.data.labels = [];
                chart.data.datasets[0].data = [];
                chart.update();
            }
        }
    } catch (err) {
        console.error("İstatistikler sıfırlanamadı:", err);
    }
});

// App Initialization
window.addEventListener('DOMContentLoaded', () => {
    initChart();
    resizeCanvas();
    // Initialize Lucide Icons
    lucide.createIcons();
});

// Re-evaluate canvas boundaries on window resizing
window.addEventListener('resize', resizeCanvas);


### ⚡ 3. Sistemi Çalıştırma
Aşağıdaki hücreyi çalıştırdığınızda Flask backend sunucusu çalışacaktır.

> [!IMPORTANT]
> Hücre çıktısında görüntülenecek **mavi renkli proxy bağlantı adresine** tıklayarak güvenlik panelinizi açın.

In [ ]:
from google.colab.output import eval_js
print("Sistem Başlatılıyor...")
print("Lütfen Güvenlik Panelini Açmak İçin Aşağıdaki Bağlantıya Tıklayın:")
print("=" * 75)
print(eval_js("google.colab.kernel.proxyPort(5000)"))
print("=" * 75)

# Flask uygulamasını çalıştır
!python app.py